In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("andrewmvd/steam-reviews")
print("Path to dataset files:", path)
print(os.listdir(path))

In [ ]:
import pandas as pd

csv_path = os.path.join(path, "dataset.csv")   

cols = ["app_id", "app_name", "review_text", "review_score", "review_votes"]
df = pd.read_csv(csv_path, usecols=cols)

In [ ]:
PLACEHOLDER = "Early Access Review" #Note: Placeholder for some reviews that have this text in data set to note EA review
n_raw = len(df)

text = df["review_text"].str.strip() # Get rid of empty text
keep = text.notna()  & (text != "") & (text != PLACEHOLDER) #Clean reviews, removing nulls, empty white space and our place holder reviews.
df = df[keep]

del text
n_valid = len(df)

seed = 42
sample_df = df.sample(n = 200000, random_state = seed) #make sure we use a seed for reproducibility

In [ ]:
cleaned_df = sample_df.copy()
cleaned_df["review_text"] = cleaned_df["review_text"].str.strip()

# dedupe on lowercased text: TF-IDF lowercases, so "Good" and "good" would be identical twins
cleaned_df["text_key"] = cleaned_df["review_text"].str.lower()
cleaned_df = cleaned_df.drop_duplicates(subset="text_key").drop(columns="text_key")
cleaned_df["review_length"] = cleaned_df["review_text"].str.len()

print("placeholders left:", (cleaned_df["review_text"] == PLACEHOLDER).sum())   # must be 0

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

seed = 42
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR, RESULTS_DIR, FIGURES_DIR = ROOT / "data", ROOT / "results", ROOT / "figures"
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

In [ ]:
cleaned_df.to_csv(DATA_DIR / "cleaned_data_sample.csv")